In [ ]:
from wrapper import BasicEnvironmentRGB
import gymnasium as gym
from gymnasium.wrappers import FrameStackObservation, TransformObservation
from matplotlib import pyplot as plt
from rltesting.torch_rl.utils import random_sample_single_env
from rltesting.torch_rl.buffers import ReplayBuffer
import numpy as np
import torch
from torch import optim
import skdim
from rltesting.utils.torch_utils import to_numpy
from rltesting.utils.logger import Logger
from rltesting.torch_rl.ppo.ppo import PPONetwork, collect_rollout, layer_init

seed = 0
framestack = 2
latent_size = 8192
obs_shape = 64
steps = 10000
lr = 1e-4

torch.manual_seed(seed)
np.random.seed(seed)    
device = ('cuda' if torch.cuda.is_available() else 'cpu')

env = gym.make('Acrobot-v1', render_mode="rgb_array")
env = BasicEnvironmentRGB(env, (obs_shape, obs_shape))
env = FrameStackObservation(env, stack_size=framestack)
env = TransformObservation(env, lambda obs: np.transpose(obs, (1, 2, 0, 3)), observation_space=gym.spaces.Box(0, 255, (framestack, 3, obs_shape, obs_shape), dtype=np.uint8))
env = TransformObservation(env, lambda obs: np.reshape(obs, (obs_shape, obs_shape, 3 * framestack)), observation_space=gym.spaces.Box(0, 255, (obs_shape, obs_shape, 3 * framestack), dtype=np.uint8))


def anneal_lr(optim, lr, update, num_updates):
    frac = 1.0 - (update) / num_updates
    new_lr = frac * lr
    optim.param_groups[0]["lr"] = new_lr

with torch.device(device):
    feature_model = AtariFeatureModel(num_hidden=3)
    target_model = AtariFeatureModel(num_hidden=0)
    rnd_opt = optim.Adam(feature_model.parameters(), lr=lr)
    rnd = RND(feature_model, target_model)

    policy = AtariPolicyNetwork(env.action_space.n)
    value = AtariRNDValueNetwork()
    ppo_network = PPONetwork(policy, value)
    for module in ppo_network.modules():
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
            layer_init(module)
    for module in rnd.modules():
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
            layer_init(module)
    ppo_opt = optim.Adam(ppo_network.parameters(), lr=args.lr, eps=1e-5)

obs = env.reset()
for _ in range(50):
    action = [env.action_space.sample() for _ in range(args.num_envs)]
    obs, _, _, _ = env.step(action)
    obs_rms.update(obs) #

total_updates = args.timesteps // (args.num_envs * args.rollout_length)
obs = obs
logger = Logger(f'logs/{args.env}')
for i in range(total_updates):
    rollout, obs = collect_rollout(env, ppo_network, rnd, args.rollout_length, obs, obs_rms)
    metrics = train_rnd(rollout, ppo_network, rnd, obs_rms, rnd_rms, ppo_opt, rnd_opt, args.num_minibatches, args.num_epochs, device, rnd_coef=1,
                        max_grad_norm=args.max_grad_norm, clip=args.clip, ent_coef=args.ent_coef, val_coef=args.val_coef, sample_prob=32 / args.num_envs)
    anneal_lr(ppo_opt, args.lr, i, total_updates)
    logger.add_metrics(metrics)
    train_rew = np.sum(rollout.rewards) / max(1, np.sum(rollout.dones))
    print(f'{i * args.num_envs * args.rollout_length}: {train_rew}')
    logger.add_scalar('train_reward', train_rew)
    if i % 50 == 0:
        eval_reward, frames = eval(ppo_network, rnd, eval_env)
        print(f'epoch {i}: {np.mean(eval_reward)}')
        logger.add_scalar('eval_reward', eval_reward)
        imageio.mimwrite(f'gifs/{args.env}_{i}.gif', frames[::4], loop=0, fps=20)
    logger.write(i * args.num_envs * args.rollout_length)
    